# FX OANDA Training (Simple)

> Designed for **VS Code + Colab-backed kernel** (CUDA), with **Google Drive persistence**.

> This trains a simple sequence model on recent OANDA candles for a hardcoded list of FX pairs, saves raw data + checkpoints to Drive, and can resume training next session.

In [22]:
from __future__ import annotations

import os
import sys
import time
from pathlib import Path

print("Python:", sys.version.split()[0])
print("CWD:", os.getcwd())

# Hardcoded training universe (edit here)
PAIRS = ["EUR_USD", "GBP_USD", "USD_JPY", "AUD_USD"]
GRANULARITY = "M5"  # OANDA v20 granularity, e.g. S5, M1, M5, M15, H1
CANDLES_PER_PAIR = 5000  # OANDA may cap counts; adjust if needed

RUN_NAME = "fx_oanda_seq"
SEED = 1337

# Training config (small + GPU-friendly)
SEQ_LEN = 128
BATCH_SIZE = 256
EPOCHS = 10
LR = 3e-4
WEIGHT_DECAY = 1e-2

# Mixed precision on GPU
USE_AMP = True
NUM_WORKERS = 2
PIN_MEMORY = True

Python: 3.12.12
CWD: /content/ml_engine


## 1) (Recommended) Mount Google Drive for persistence
This is what makes checkpoints + cached candles survive runtime resets/sessions.

In [23]:
from __future__ import annotations

from pathlib import Path
import threading

# Set True if you want Drive persistence (will attempt an auth flow).
MOUNT_DRIVE = True

def mount_drive(max_wait_s: int = 8) -> Path | None:
    try:
        from google.colab import drive  # type: ignore
    except Exception:
        print("Not running in a Colab environment; skipping Drive mount.")
        return None
    if not MOUNT_DRIVE:
        print("Drive mount skipped (MOUNT_DRIVE=False).")
        return None

    done = {"ok": False, "err": None}

    def _worker() -> None:
        try:
            # Small timeout so this cell doesn't hang in VS Code kernels.
            drive.mount("/content/drive", timeout_ms=10_000)
            done["ok"] = True
        except Exception as e:
            done["err"] = e

    t = threading.Thread(target=_worker, daemon=True)
    t.start()
    t.join(timeout=max_wait_s)
    if t.is_alive():
        print("⚠️ Drive mount still waiting for auth.")
        print("Open the notebook in the Colab UI to complete Drive auth, then rerun this cell.")
        return None
    if done["err"] is not None:
        print(f"⚠️ Drive mount failed: {done['err']}")
        return None
    if not done["ok"]:
        print("⚠️ Drive mount did not complete.")
        return None

    mydrive = Path("/content/drive/MyDrive")
    root = mydrive if mydrive.exists() else Path("/content/drive")
    print("✓ Drive mounted")
    return root

DRIVE_ROOT = mount_drive()
print("DRIVE_ROOT:", DRIVE_ROOT)

ARTIFACTS_ROOT = (DRIVE_ROOT / "ml_engine_artifacts") if DRIVE_ROOT else (Path("/content") / "ml_engine_artifacts")
ARTIFACTS_ROOT.mkdir(parents=True, exist_ok=True)
print("ARTIFACTS_ROOT:", ARTIFACTS_ROOT)

RUN_DIR = ARTIFACTS_ROOT / RUN_NAME
DATA_DIR = RUN_DIR / "data"
CKPT_DIR = RUN_DIR / "checkpoints"
DATA_DIR.mkdir(parents=True, exist_ok=True)
CKPT_DIR.mkdir(parents=True, exist_ok=True)
print("RUN_DIR:", RUN_DIR)

⚠️ Drive mount still waiting for auth.
Open the notebook in the Colab UI to complete Drive auth, then rerun this cell.
DRIVE_ROOT: None
ARTIFACTS_ROOT: /content/ml_engine_artifacts
RUN_DIR: /content/ml_engine_artifacts/fx_oanda_seq


## 2) Clone the repo into the kernel filesystem
This makes `oanda_practice.py` available in the Colab kernel.

In [24]:
import subprocess
from pathlib import Path
import os
import sys

REPO_URL = "https://github.com/Raynergy-svg/ml_engine.git"
REPO_DIR = Path("/content/ml_engine")

def sh(cmd: list[str], cwd: Path | None = None) -> None:
    print("$", " ".join(cmd))
    subprocess.check_call(cmd, cwd=str(cwd) if cwd else None)

if not REPO_DIR.exists():
    sh(["git", "clone", "--depth", "1", REPO_URL, str(REPO_DIR)])
else:
    if (REPO_DIR / ".git").exists():
        sh(["git", "fetch", "origin"], cwd=REPO_DIR)
        sh(["git", "reset", "--hard", "origin/main"], cwd=REPO_DIR)

os.chdir(str(REPO_DIR))
print("Repo cwd:", os.getcwd())

# Make imports work even if cwd changes later
if str(REPO_DIR) not in sys.path:
    sys.path.insert(0, str(REPO_DIR))

$ git fetch origin
$ git reset --hard origin/main
Repo cwd: /content/ml_engine
$ git reset --hard origin/main
Repo cwd: /content/ml_engine


## 3) Install minimal dependencies
Colab usually already has CUDA-enabled `torch`; we only ensure `python-dotenv` + data stack.

In [25]:
import sys
import subprocess

def pip_install(pkgs: list[str]) -> None:
    cmd = [sys.executable, "-m", "pip", "install", "-q", "--no-cache-dir"] + pkgs
    print("$", " ".join(cmd))
    subprocess.check_call(cmd)

# Torch is usually present in Colab with CUDA; don't override unless missing.
try:
    import torch  # type: ignore
    print("torch:", torch.__version__)
except Exception:
    pip_install(["torch", "torchvision", "torchaudio"])
    import torch  # type: ignore
    print("torch:", torch.__version__)

pip_install(["python-dotenv>=1.0.0", "pandas>=2.0.0", "numpy>=1.24.0"])

torch: 2.9.0+cpu
$ /usr/bin/python3 -m pip install -q --no-cache-dir python-dotenv>=1.0.0 pandas>=2.0.0 numpy>=1.24.0


## 4) OANDA credentials (practice)
**Note:** This notebook runs on a remote kernel (Colab). It **cannot** see files on your local computer (like your local `.env`).
We will look for `.env` in:
1. The cloned repo (`/content/ml_engine`)
2. Google Drive (if mounted)
3. Or prompt you to enter them manually.

In [26]:
import os
from pathlib import Path

# Enable prompting if keys are missing
PROMPT_FOR_SECRETS = True

def load_dotenv_from_repo() -> tuple[str, list[str]]:
    try:
        from dotenv import load_dotenv  # type: ignore
    except Exception:
        return "", []
    
    # Search locations: CWD, Repo Dir, and Drive (if mounted)
    repo_dir = globals().get("REPO_DIR")
    drive_root = globals().get("DRIVE_ROOT")
    
    bases = [Path.cwd()]
    if repo_dir:
        bases.append(Path(repo_dir))
    if drive_root:
        # Look in Drive root and a common 'ml_engine' folder in Drive
        bases.append(drive_root)
        bases.append(drive_root / "ml_engine")

    seen = set()
    searched = []
    
    for base in bases:
        base = Path(base).resolve()
        if str(base) in seen:
            continue
        seen.add(str(base))
        searched.append(str(base))
        
        # Prefer .env, then .env.local
        for name in (".env", ".env.local"):
            p = base / name
            if p.exists():
                load_dotenv(p, override=False)
                return str(p), searched
    return "", searched

loaded, searched = load_dotenv_from_repo()
if loaded:
    print("Loaded:", loaded)
else:
    print("No .env/.env.local found in:", searched)

def maybe_prompt() -> None:
    # Check if keys are already present
    if (os.getenv("OANDA_API_TOKEN") or os.getenv("OANDA_API_KEY")) and os.getenv("OANDA_ACCOUNT_ID"):
        return
        
    if not PROMPT_FOR_SECRETS:
        print("Secrets missing and prompting disabled.")
        return

    print("\n--- Enter OANDA Credentials (hidden) ---")
    import getpass
    token = getpass.getpass("OANDA_API_TOKEN: ").strip()
    if token:
        os.environ["OANDA_API_TOKEN"] = token
    
    account = input("OANDA_ACCOUNT_ID: ").strip()
    if account:
        os.environ["OANDA_ACCOUNT_ID"] = account

maybe_prompt()

print("Token set:", bool(os.getenv("OANDA_API_TOKEN") or os.getenv("OANDA_API_KEY")))
print("Account set:", bool(os.getenv("OANDA_ACCOUNT_ID")))

No .env/.env.local found in: ['/content/ml_engine']
Token set: True
Account set: True


## 5) Download/cache candles (per pair)
We fetch the most recent `CANDLES_PER_PAIR` candles for each pair and cache to Drive/Artifacts so the next run can resume fast.

In [27]:
import pandas as pd
import numpy as np

from oanda_practice import OandaPracticeClient

def candles_to_df(payload: dict, pair: str) -> pd.DataFrame:
    candles = payload.get("candles") or []
    rows = []
    for c in candles:
        if not isinstance(c, dict) or not c.get("complete", True):
            continue
        mid = c.get("mid") or {}
        rows.append(
            {
                "pair": pair,
                "time": c.get("time"),
                "o": float(mid.get("o")),
                "h": float(mid.get("h")),
                "l": float(mid.get("l")),
                "c": float(mid.get("c")),
                "v": int(c.get("volume", 0)),
            }
        )
    df = pd.DataFrame(rows)
    if df.empty:
        return df
    df["time"] = pd.to_datetime(df["time"], utc=True, errors="coerce")
    df = df.dropna(subset=["time"]).sort_values(["pair", "time"]).reset_index(drop=True)
    return df

client = OandaPracticeClient.from_env()

all_dfs = []
for pair in PAIRS:
    out_path = DATA_DIR / f"candles_{pair}_{GRANULARITY}_{CANDLES_PER_PAIR}.parquet"
    if out_path.exists():
        print("Cache hit:", out_path.name)
        all_dfs.append(pd.read_parquet(out_path))
        continue
    print("Fetching:", pair, GRANULARITY, CANDLES_PER_PAIR)
    payload = client.get_candles(pair, granularity=GRANULARITY, count=int(CANDLES_PER_PAIR), price="M")
    df = candles_to_df(payload, pair)
    if df.empty:
        raise RuntimeError(f"No candles returned for {pair}")
    df.to_parquet(out_path, index=False)
    print("Saved:", out_path)
    all_dfs.append(df)

candles_df = pd.concat(all_dfs, ignore_index=True)
print("Rows:", len(candles_df))
candles_df.head()

Cache hit: candles_EUR_USD_M5_5000.parquet
Cache hit: candles_GBP_USD_M5_5000.parquet
Cache hit: candles_USD_JPY_M5_5000.parquet
Cache hit: candles_AUD_USD_M5_5000.parquet
Rows: 20000


,pair,time,o,h,l,c,v
0,EUR_USD,2025-11-19 13:20:00+00:00,1.15893,1.15910,1.15854,1.15856,599
1,EUR_USD,2025-11-19 13:25:00+00:00,1.15856,1.15867,1.15834,1.15835,366
2,EUR_USD,2025-11-19 13:30:00+00:00,1.15836,1.15840,1.15786,1.15790,475
3,EUR_USD,2025-11-19 13:35:00+00:00,1.15791,1.15792,1.15764,1.15778,489
4,EUR_USD,2025-11-19 13:40:00+00:00,1.15778,1.15782,1.15746,1.15746,356


## 6) Build a GPU-friendly training dataset
We predict next-step log return from a rolling window of past log returns (per pair).

In [28]:
import math
import random
import numpy as np
import pandas as pd
import torch
from torch.utils.data import Dataset, DataLoader
from sklearn.preprocessing import RobustScaler

def set_seed(seed: int) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.benchmark = True
    torch.backends.cuda.matmul.allow_tf32 = True
    torch.backends.cudnn.allow_tf32 = True

set_seed(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)
if device.type == "cuda":
    print("GPU:", torch.cuda.get_device_name(0))

# ---------------------------------------------------------
# Feature Engineering for UnifiedMarketNet
# Input Features (5):
# 0. Log Return (Close)
# 1. Log Range (High / Low)
# 2. Log Body (Close / Open)
# 3. Log Volume (Volume + 1)
# 4. Volatility (Rolling StdDev of Log Returns)
# ---------------------------------------------------------

candles_df = candles_df.sort_values(["pair", "time"]).reset_index(drop=True)

# 1. Log Return
candles_df["log_c"] = np.log(candles_df["c"].astype(float))
candles_df["log_ret"] = candles_df.groupby("pair")["log_c"].diff().fillna(0.0)

# 2. Log Range (High/Low)
candles_df["log_hl"] = np.log(candles_df["h"].astype(float) / candles_df["l"].astype(float)).fillna(0.0)

# 3. Log Body (Close/Open)
candles_df["log_co"] = np.log(candles_df["c"].astype(float) / candles_df["o"].astype(float)).fillna(0.0)

# 4. Log Volume
candles_df["log_vol"] = np.log1p(candles_df["v"].astype(float)).fillna(0.0)

# 5. Rolling Volatility (window=20)
candles_df["volatility"] = candles_df.groupby("pair")["log_ret"].rolling(window=20).std().reset_index(0, drop=True).fillna(0.0)

# Drop initial NaNs from rolling
candles_df = candles_df.dropna().reset_index(drop=True)

# ---------------------------------------------------------
# Target Generation (Multi-Task)
# 1. Price: Next Log Return
# 2. Trend: 1 if Next Log Return > 0 else 0
# 3. Risk: Next Volatility (proxy)
# 4. State: Market Regime (0=Low Vol, 1=High Vol) - simplified
# ---------------------------------------------------------

# Shift for targets
candles_df["target_price"] = candles_df.groupby("pair")["log_ret"].shift(-1)
candles_df["target_risk"] = candles_df.groupby("pair")["volatility"].shift(-1) # Predict next volatility
candles_df = candles_df.dropna().reset_index(drop=True)

# Binary Trend
candles_df["target_trend"] = (candles_df["target_price"] > 0).astype(float)

# Binary State (Regime) - Split by median volatility per pair
pair_median_vol = candles_df.groupby("pair")["volatility"].median()
def get_state(row):
    return 1.0 if row["volatility"] > pair_median_vol[row["pair"]] else 0.0
candles_df["target_state"] = candles_df.apply(get_state, axis=1)

# ---------------------------------------------------------
# Splitting & Scaling
# ---------------------------------------------------------
pairs = sorted(candles_df["pair"].unique().tolist())
pair_to_idx = {p: i for i, p in enumerate(pairs)}
candles_df["pair_id"] = candles_df["pair"].map(pair_to_idx).astype(int)

train_parts = []
val_parts = []

# We need to save scalers for the Buddy app
scalers = {} 

feature_cols = ["log_ret", "log_hl", "log_co", "log_vol", "volatility"]

for p in pairs:
    d = candles_df[candles_df["pair"] == p].reset_index(drop=True)
    n = len(d)
    cut = int(n * 0.9)
    
    train_d = d.iloc[:cut].copy()
    val_d = d.iloc[cut:].copy()
    
    # Fit RobustScaler on Train
    sc = RobustScaler()
    train_d[feature_cols] = sc.fit_transform(train_d[feature_cols])
    val_d[feature_cols] = sc.transform(val_d[feature_cols])
    
    scalers[p] = sc
    
    train_parts.append(train_d)
    val_parts.append(val_d)

train_df = pd.concat(train_parts, ignore_index=True)
val_df = pd.concat(val_parts, ignore_index=True)

# ---------------------------------------------------------
# Dataset
# ---------------------------------------------------------
class UnifiedTimeseriesDataset(Dataset):
    def __init__(self, df: pd.DataFrame, seq_len: int):
        self.seq_len = seq_len
        self.samples = []
        
        cols = ["log_ret", "log_hl", "log_co", "log_vol", "volatility"]
        target_cols = ["target_price", "target_trend", "target_risk", "target_state"]
        
        for p in df["pair"].unique():
            d = df[df["pair"] == p].reset_index(drop=True)
            if len(d) <= seq_len:
                continue
                
            X_all = d[cols].values.astype(np.float32)
            Y_all = d[target_cols].values.astype(np.float32)
            p_id = d["pair_id"].iloc[0]
            
            # Create sliding windows
            # We need X[i-seq_len : i] to predict Y[i] (which is already shifted in dataframe)
            # Actually, in dataframe: target_price at row i is the return for i+1.
            # So if we input rows [i-seq_len+1 ... i], we predict target at i.
            
            for i in range(seq_len, len(d)):
                x_window = X_all[i-seq_len : i]
                y_targets = Y_all[i-1] # The target for the last step in the window
                # Wait, if target_price is shifted(-1), then at row i, target_price is return(i+1).
                # If we feed [0..seq_len-1], we want to predict return(seq_len).
                # Row (seq_len-1) has target_price = return(seq_len).
                # So we take X[i-seq_len : i] and Y[i-1].
                
                self.samples.append((x_window, y_targets, p_id))

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        x, y, pid = self.samples[idx]
        # y is [price, trend, risk, state]
        return (
            torch.from_numpy(x),          # [Seq, 5]
            torch.tensor(pid, dtype=torch.long),
            torch.from_numpy(y)           # [4]
        )

train_ds = UnifiedTimeseriesDataset(train_df, seq_len=SEQ_LEN)
val_ds = UnifiedTimeseriesDataset(val_df, seq_len=SEQ_LEN)
print(f"Train samples: {len(train_ds)}, Val samples: {len(val_ds)}")

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, num_workers=NUM_WORKERS, pin_memory=PIN_MEMORY)
val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS, pin_memory=PIN_MEMORY)


Device: cpu
Train samples: 17484, Val samples: 1488
Train samples: 17484, Val samples: 1488


## 7) Model + training loop (AMP on GPU)
This is a small GRU-based forecaster with a per-pair embedding. It saves `latest.pt` each epoch and resumes automatically if present.

In [29]:
import torch
import torch.nn as nn
import torch.nn.functional as F

# Try to import UnifiedMarketNet from the cloned repo
try:
    from ml_engine.neural_engine_unified import UnifiedMarketNet
    print("Successfully imported UnifiedMarketNet from ml_engine.")
except ImportError:
    print("Could not import UnifiedMarketNet. Defining a compatible version inline.")
    # Inline definition if import fails (fallback)
    class UnifiedMarketNet(nn.Module):
        def __init__(self, input_dim=5, hidden_dim=128, n_layers=2, dropout=0.2):
            super().__init__()
            self.lstm = nn.LSTM(input_dim, hidden_dim, num_layers=n_layers, 
                              batch_first=True, dropout=dropout)
            
            # 4 Heads
            self.head_price = nn.Sequential(
                nn.Linear(hidden_dim, 64), nn.GELU(), nn.Linear(64, 1)
            )
            self.head_trend = nn.Sequential(
                nn.Linear(hidden_dim, 64), nn.GELU(), nn.Linear(64, 1), nn.Sigmoid()
            )
            self.head_risk = nn.Sequential(
                nn.Linear(hidden_dim, 64), nn.GELU(), nn.Linear(64, 1), nn.Softplus()
            )
            self.head_state = nn.Sequential(
                nn.Linear(hidden_dim, 64), nn.GELU(), nn.Linear(64, 1), nn.Sigmoid()
            )
            
        def forward(self, x):
            # x: [B, Seq, Feat]
            out, _ = self.lstm(x)
            last = out[:, -1, :]
            
            price = self.head_price(last)
            trend = self.head_trend(last)
            risk = self.head_risk(last)
            state = self.head_state(last)
            
            return price, trend, risk, state

# Initialize Model
# Note: Buddy expects specific input_dim=5 usually
model = UnifiedMarketNet(input_dim=5, hidden_dim=128, n_layers=2, dropout=0.2).to(device)

opt = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
scaler = torch.cuda.amp.GradScaler(enabled=(USE_AMP and device.type == "cuda"))

# Checkpoint Loading
CKPT_PATH = CKPT_DIR / "unified_latest.pt"
start_epoch = 0
best_val_loss = float("inf")

if CKPT_PATH.exists():
    try:
        ckpt = torch.load(CKPT_PATH, map_location="cpu")
        model.load_state_dict(ckpt["model_state_dict"])
        opt.load_state_dict(ckpt["optimizer_state_dict"])
        start_epoch = ckpt.get("epoch", 0) + 1
        best_val_loss = ckpt.get("best_val_loss", float("inf"))
        print(f"Resumed from {CKPT_PATH} (epoch={start_epoch})")
    except Exception as e:
        print(f"Failed to resume: {e}")
else:
    print("Starting fresh training.")

# Loss Functions
criterion_price = nn.MSELoss()
criterion_trend = nn.BCELoss()
criterion_risk = nn.MSELoss()
criterion_state = nn.BCELoss()

def compute_loss(preds, targets):
    # preds: (price, trend, risk, state)
    # targets: [B, 4] -> price, trend, risk, state
    
    p_pred, t_pred, r_pred, s_pred = preds
    
    t_price = targets[:, 0].unsqueeze(1)
    t_trend = targets[:, 1].unsqueeze(1)
    t_risk  = targets[:, 2].unsqueeze(1)
    t_state = targets[:, 3].unsqueeze(1)
    
    l_price = criterion_price(p_pred, t_price)
    l_trend = criterion_trend(t_pred, t_trend)
    l_risk  = criterion_risk(r_pred, t_risk)
    l_state = criterion_state(s_pred, t_state)
    
    # Weighted sum
    total_loss = l_price + 0.5*l_trend + 0.5*l_risk + 0.5*l_state
    return total_loss, (l_price, l_trend, l_risk, l_state)

def run_eval():
    model.eval()
    losses = []
    with torch.no_grad():
        for xb, pid, yb in val_loader:
            xb = xb.to(device, non_blocking=True)
            yb = yb.to(device, non_blocking=True)
            
            with torch.cuda.amp.autocast(enabled=(USE_AMP and device.type == "cuda")):
                preds = model(xb)
                loss, _ = compute_loss(preds, yb)
            losses.append(loss.item())
    return np.mean(losses) if losses else float("inf")

# Training Loop
for epoch in range(start_epoch, EPOCHS):
    model.train()
    t0 = time.time()
    running_loss = 0.0
    n_batches = 0
    
    for xb, pid, yb in train_loader:
        xb = xb.to(device, non_blocking=True)
        yb = yb.to(device, non_blocking=True)
        
        opt.zero_grad(set_to_none=True)
        
        with torch.cuda.amp.autocast(enabled=(USE_AMP and device.type == "cuda")):
            preds = model(xb)
            loss, components = compute_loss(preds, yb)
            
        scaler.scale(loss).backward()
        scaler.step(opt)
        scaler.update()
        
        running_loss += loss.item()
        n_batches += 1
        
    avg_train_loss = running_loss / max(1, n_batches)
    val_loss = run_eval()
    dt = time.time() - t0
    
    print(f"Epoch {epoch+1}/{EPOCHS} | Train: {avg_train_loss:.5f} | Val: {val_loss:.5f} | Time: {dt:.1f}s")
    
    # Save Checkpoint (Buddy Compatible Format)
    if val_loss < best_val_loss:
        best_val_loss = val_loss
        
    save_dict = {
        "epoch": epoch,
        "model_state_dict": model.state_dict(),
        "optimizer_state_dict": opt.state_dict(),
        "best_val_loss": best_val_loss,
        "config": {
            "input_dim": 5,
            "hidden_dim": 128,
            "n_layers": 2,
            "dropout": 0.2,
            "seq_len": SEQ_LEN,
            "pairs": PAIRS
        },
        "metadata": {
            "scalers": scalers,  # The RobustScalers fitted in Cell 6
            "pairs": PAIRS,
            "granularity": GRANULARITY
        },
        "model_arch": "UnifiedMarketNet"
    }
    
    torch.save(save_dict, CKPT_PATH)
    print(f"Saved checkpoint to {CKPT_PATH}")


Could not import UnifiedMarketNet. Defining a compatible version inline.
Starting fresh training.


/tmp/ipython-input-4102051596.py:49: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = torch.cuda.amp.GradScaler(enabled=(USE_AMP and device.type == "cuda"))
/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:668: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  warnings.warn(warn_msg)
/tmp/ipython-input-4102051596.py:122: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(enabled=(USE_AMP and device.type == "cuda")):
/tmp/ipython-input-4102051596.py:122: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(enabled=(USE_AMP and device.type == "cuda")):
/tmp/ipython-input-4102051596.py:103: FutureWarning: `to

Epoch 1/10 | Train: 0.73688 | Val: 0.52273 | Time: 110.8s
Saved checkpoint to /content/ml_engine_artifacts/fx_oanda_seq/checkpoints/unified_latest.pt
Epoch 2/10 | Train: 0.45177 | Val: 0.43339 | Time: 107.6s
Saved checkpoint to /content/ml_engine_artifacts/fx_oanda_seq/checkpoints/unified_latest.pt
Epoch 3/10 | Train: 0.42219 | Val: 0.41935 | Time: 105.9s
Saved checkpoint to /content/ml_engine_artifacts/fx_oanda_seq/checkpoints/unified_latest.pt
Epoch 4/10 | Train: 0.41026 | Val: 0.41288 | Time: 107.3s
Saved checkpoint to /content/ml_engine_artifacts/fx_oanda_seq/checkpoints/unified_latest.pt
Epoch 5/10 | Train: 0.40373 | Val: 0.40477 | Time: 106.9s
Saved checkpoint to /content/ml_engine_artifacts/fx_oanda_seq/checkpoints/unified_latest.pt
Epoch 6/10 | Train: 0.39703 | Val: 0.41092 | Time: 107.5s
Saved checkpoint to /content/ml_engine_artifacts/fx_oanda_seq/checkpoints/unified_latest.pt
Epoch 7/10 | Train: 0.39313 | Val: 0.39382 | Time: 105.8s
Saved checkpoint to /content/ml_engine_art

## 8) What gets saved / how to resume
- Cached candles: `RUN_DIR/data/*.parquet`
- Checkpoint (auto-resume): `RUN_DIR/checkpoints/unified_latest.pt`

This checkpoint is now compatible with the **Buddy** application.
It contains:
1. `model_state_dict`: Weights for `UnifiedMarketNet`
2. `metadata`: Contains `scalers` (RobustScaler) for each pair.
3. `config`: Model hyperparameters.

To use in Buddy:
1. Download `unified_latest.pt` from Google Drive.
2. Place it in your local `trained_data/models/` or `trained_data/checkpoints/`.
3. Run Buddy and point it to this checkpoint.
